# 3 · Temas avanzados: RL multiagente con PettingZoo

*Sección del video 2:34 – fin.*

Hasta ahora había **un** agente y el entorno era "naturaleza". En muchos problemas reales
hay varios agentes que deciden a la vez (subastas, tráfico, juegos, robots en equipo).
Cada agente ve a los demás como parte del entorno, y ese entorno **cambia** mientras los
otros aprenden → es más difícil (no estacionario).

**PettingZoo** (también de la Farama Foundation) es "Gymnasium para multiagente". Tiene dos APIs:
- **AEC** (*Agent Environment Cycle*): los agentes actúan **por turnos** (ajedrez, gato).
- **Parallel**: todos actúan **al mismo tiempo** (piedra-papel-tijera, simulaciones).

In [1]:
import numpy as np
from pettingzoo.classic import tictactoe_v3, rps_v2

## API AEC: Gato (tic-tac-toe)
El patrón es: `for agent in env.agent_iter()` → `env.last()` te da la observación del agente
que le toca → `env.step(accion)`.

Detalle importante: la observación trae una **máscara de acciones** (qué casillas están libres).
Un agente que respeta la máscara nunca hace jugadas ilegales.

In [2]:
env = tictactoe_v3.env()
env.reset(seed=42)

for agente in env.agent_iter():
    obs, reward, terminated, truncated, info = env.last()
    if terminated or truncated:
        print(agente, "recompensa final:", reward)
        accion = None
    else:
        accion = env.action_space(agente).sample(obs["action_mask"])  # jugada al azar pero legal
        print(agente, "juega en la casilla", accion)
    env.step(accion)

player_1 juega en la casilla 6
player_2 juega en la casilla 5
player_1 juega en la casilla 2
player_2 juega en la casilla 1
player_1 juega en la casilla 0
player_2 juega en la casilla 8
player_1 juega en la casilla 4
player_2 recompensa final: -1
player_1 recompensa final: 1


## Muchas partidas aleatorias
Con dos jugadores aleatorios, ¿cuánto gana quien sale primero?

In [3]:
def partida(seed):
    env = tictactoe_v3.env()
    env.reset(seed=seed)
    for agente in env.agent_iter():
        obs, reward, terminated, truncated, info = env.last()
        if terminated or truncated:
            if agente == "player_1":
                resultado = reward
            accion = None
        else:
            accion = env.action_space(agente).sample(obs["action_mask"])
        env.step(accion)
    return resultado

resultados = np.array([partida(s) for s in range(2000)])
print("Gana el primero:", np.mean(resultados == 1))
print("Empate:         ", np.mean(resultados == 0))
print("Pierde:         ", np.mean(resultados == -1))

Gana el primero: 0.5845
Empate:          0.1165
Pierde:          0.299


## API Parallel: piedra, papel o tijera
Aquí `step` recibe un **diccionario** {agente: acción} y regresa diccionarios.

In [4]:
env = rps_v2.parallel_env(max_cycles=10)
env.reset(seed=0)
nombres = ["piedra", "papel", "tijera"]

while env.agents:
    acciones = {a: env.action_space(a).sample() for a in env.agents}
    obs, rewards, terminated, truncated, info = env.step(acciones)
    print(nombres[acciones["player_0"]], "vs", nombres[acciones["player_1"]], "→", dict(rewards))

papel vs tijera → {'player_0': -1, 'player_1': 1}
piedra vs papel → {'player_0': -1, 'player_1': 1}
papel vs piedra → {'player_0': 1, 'player_1': -1}
tijera vs papel → {'player_0': 1, 'player_1': -1}
piedra vs tijera → {'player_0': 1, 'player_1': -1}
piedra vs papel → {'player_0': -1, 'player_1': 1}
piedra vs tijera → {'player_0': 1, 'player_1': -1}
piedra vs piedra → {'player_0': 0, 'player_1': 0}
piedra vs piedra → {'player_0': 0, 'player_1': 0}
piedra vs tijera → {'player_0': 1, 'player_1': -1}


## Para llevar
- Multiagente = cada agente optimiza su propia recompensa, pero las acciones de los demás
  forman parte de su "información exógena" (en términos de Powell, $W_{t+1}$ ya no es
  naturaleza aleatoria fija: es otra política que también aprende).
- Las técnicas que vimos (Q-learning, DQN) se pueden aplicar a cada agente por separado
  ("independent learners"), aunque hay métodos especializados (self-play, MADDPG, QMIX…).
- PettingZoo da la interfaz estándar; librerías como Stable-Baselines3, RLlib o AgileRL
  ponen los algoritmos.